In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'deepspeed==0.17.4' -q
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

# Triplet

In [ ]:
%%writefile triplet.py
#!/usr/bin/env python3
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import random
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
from sklearn.metrics.pairwise import cosine_similarity
import re
from urllib.parse import urlparse
# import faiss
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    # Regex pattern to match URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]

            # Extract meaningful path parts (first 1-2 segments)
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                # Take first 1-2 meaningful path segments
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text))


def load_test_data():
    """Load test data."""
    print("Loading test data...")
    test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
    print(f"Loaded {len(test_df)} test examples")
    print(f"Unique rules: {test_df['rule'].nunique()}")
    return test_df


def collect_all_texts(test_df):
    """Collect all unique texts from test set."""
    print("\nCollecting all texts for embedding...")
    
    all_texts = set()
    
    # Add all bodies
    for body in test_df['body']:
        if pd.notna(body):
            all_texts.add(cleaner(str(body)))
    
    # Add all positive and negative examples
    example_cols = ['positive_example_1', 'positive_example_2', 
                   'negative_example_1', 'negative_example_2']
    
    for col in example_cols:
        for example in test_df[col]:
            if pd.notna(example):
                all_texts.add(cleaner(str(example)))
    
    all_texts = list(all_texts)
    print(f"Collected {len(all_texts)} unique texts")
    return all_texts


def generate_embeddings(texts, model, batch_size=64):
    """Generate BGE embeddings for all texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    
    embeddings = model.encode(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True
    )
    
    return embeddings


def create_test_triplet_dataset(test_df, augmentation_factor=2, random_seed=42, subsample_fraction=1.0):
    """Create triplet dataset from test data: anchor=rule, positive=positive_example, negative=negative_example."""
    random.seed(random_seed)
    np.random.seed(random_seed)
    
    anchors = []
    positives = []
    negatives = []
    
    print("Creating rule-aligned triplets from test data...")
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
        rule = cleaner(str(row['rule']))
        
        pos_examples = []  # Will contain compliant comments (rule-aligned)
        neg_examples = []  # Will contain violating comments (rule-misaligned)

        for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
            if pd.notna(row[neg_col]):
                pos_examples.append(cleaner(str(row[neg_col])))

        for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
            if pd.notna(row[pos_col]):
                neg_examples.append(cleaner(str(row[pos_col])))
        
        for pos_ex in pos_examples:
            for neg_ex in neg_examples:
                anchors.append(rule)
                positives.append(pos_ex)
                negatives.append(neg_ex)
    
    if augmentation_factor > 0:
        print(f"Adding {augmentation_factor}x augmentation...")
        
        rule_positives = {}
        rule_negatives = {}
        
        for rule in test_df['rule'].unique():
            rule_df = test_df[test_df['rule'] == rule]
            
            pos_pool = []
            neg_pool = []
            
            for _, row in rule_df.iterrows():
                for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
                    if pd.notna(row[neg_col]):
                        pos_pool.append(cleaner(str(row[neg_col])))
                for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
                    if pd.notna(row[pos_col]):
                        neg_pool.append(cleaner(str(row[pos_col])))
            
            rule_positives[rule] = list(set(pos_pool))
            rule_negatives[rule] = list(set(neg_pool))
        
        for rule in test_df['rule'].unique():
            clean_rule = cleaner(str(rule))
            pos_pool = rule_positives[rule]
            neg_pool = rule_negatives[rule]
            
            n_samples = min(augmentation_factor * len(pos_pool), len(pos_pool) * len(neg_pool))
            
            for _ in range(n_samples):
                if pos_pool and neg_pool:
                    anchors.append(clean_rule)
                    positives.append(random.choice(pos_pool))
                    negatives.append(random.choice(neg_pool))
    
    combined = list(zip(anchors, positives, negatives))
    random.shuffle(combined)
    
    # Apply subsampling if requested
    original_count = len(combined)
    if subsample_fraction < 1.0:
        n_samples = int(len(combined) * subsample_fraction)
        combined = combined[:n_samples]
        print(f"Subsampled {original_count} -> {len(combined)} triplets ({subsample_fraction*100:.1f}%)")
    
    anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
    
    print(f"Created {len(anchors)} triplets from test data")
    
    dataset = Dataset.from_dict({
        'anchor': list(anchors),
        'positive': list(positives),
        'negative': list(negatives)
    })
    
    return dataset


def fine_tune_model(model, train_dataset, epochs=3, batch_size=32, learning_rate=2e-5, margin=0.25, output_dir="./models/test-finetuned-bge"):
    """Fine-tune the sentence transformer model using triplet loss on test data."""
    
    print(f"Fine-tuning model on {len(train_dataset)} triplets...")
    
    loss = TripletLoss(model=model, triplet_margin=margin)
    
    # Calculate max_steps for small datasets
    dataset_size = len(train_dataset)
    steps_per_epoch = max(1, dataset_size // batch_size)
    max_steps = steps_per_epoch * epochs

    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=0,
        learning_rate=learning_rate,
        logging_steps=max(1, max_steps // 4),
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        max_grad_norm=1.0,
        dataloader_drop_last=False,
        gradient_checkpointing=True,
        gradient_accumulation_steps = 1,
        max_steps=max_steps,
        report_to="none"
    )
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        loss=loss,
    )
    
    trainer.train()
    
    final_model_path = f"{output_dir}/final"
    print(f"Saving fine-tuned model to {final_model_path}...")
    model.save_pretrained(final_model_path)
    
    return model, final_model_path


def load_or_create_finetuned_model(test_df):
    """Load fine-tuned model if exists, otherwise create and fine-tune it."""
    
    fine_tuned_path = "./models/test-finetuned-bge/final"
    
    if os.path.exists(fine_tuned_path):
        print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
        try:
            word_embedding_model = models.Transformer(fine_tuned_path, max_seq_length=128, do_lower_case=True)
            pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
            model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded fine-tuned model with explicit pooling")
        except:
            model = SentenceTransformer(fine_tuned_path)
            print("Loaded fine-tuned model with default configuration")
        model.half()
        return model
    
    print("Fine-tuned model not found. Creating new one...")
    
    print("Loading base BGE embedding model...")
    # Try Kaggle path first, fallback to HuggingFace
    try:
        model_path = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
        word_embedding_model = models.Transformer(model_path, max_seq_length=128, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Kaggle path with explicit pooling")
    except:
        model_path = ""  # BAAI/bge-small-en-v1.5
        word_embedding_model = models.Transformer(model_path, max_seq_length=128, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from local path with explicit pooling")
    
    
    triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=16, subsample_fraction=1.)
    
    fine_tuned_model, model_path = fine_tune_model(
        model=base_model,
        train_dataset=triplet_dataset,
        epochs=1,
        batch_size=32,
        learning_rate=2e-5,
        margin=0.25
    )
    
    print(f"Fine-tuning completed. Model saved to: {model_path}")
    fine_tuned_model.half()
    return fine_tuned_model


def generate_rule_embeddings(test_df, model):
    """Generate embeddings for each unique rule."""
    print("Generating rule embeddings...")
    
    unique_rules = test_df['rule'].unique()
    rule_embeddings = {}
    
    for rule in unique_rules:
        clean_rule = cleaner(str(rule))
        rule_emb = model.encode(
            clean_rule,
            convert_to_tensor=False,
            normalize_embeddings=True
        )
        rule_embeddings[rule] = rule_emb
        
    print(f"Generated embeddings for {len(rule_embeddings)} rules")
    return rule_embeddings


def create_rule_centroids(test_df, text_to_embedding, rule_embeddings):
    """Create single centroid (mean) for positive and negative examples for each rule."""
    print(f"\nCreating rule centroids (single mean centroid per type)...")

    rule_centroids = {}

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]

        # Collect positive examples
        pos_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        pos_embeddings.append(text_to_embedding[clean_text])

        # Collect negative examples
        neg_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        neg_embeddings.append(text_to_embedding[clean_text])

        if pos_embeddings and neg_embeddings:
            pos_embeddings = np.array(pos_embeddings)
            neg_embeddings = np.array(neg_embeddings)

            # Compute mean centroids
            pos_centroid = pos_embeddings.mean(axis=0)
            neg_centroid = neg_embeddings.mean(axis=0)

            # Normalize centroids
            pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
            neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)

            rule_centroids[rule] = {
                'positive': pos_centroid,
                'negative': neg_centroid,
                'pos_count': len(pos_embeddings),
                'neg_count': len(neg_embeddings),
                'rule_embedding': rule_embeddings[rule]
            }

            print(f"  Rule: {rule[:50]}... - Pos: {len(pos_embeddings)}, Neg: {len(neg_embeddings)}")

    print(f"Created centroids for {len(rule_centroids)} rules")
    return rule_centroids


def predict_test_set(test_df, text_to_embedding, rule_centroids):
    """Predict test set using Euclidean distance between body and pos/neg centroids."""
    print("\nMaking predictions on test set with Euclidean distance...")

    row_ids = []
    predictions = []

    for rule in test_df['rule'].unique():
        print(f"  Processing rule: {rule[:50]}...")
        rule_data = test_df[test_df['rule'] == rule]

        if rule not in rule_centroids:
            continue

        pos_centroid = rule_centroids[rule]['positive']
        neg_centroid = rule_centroids[rule]['negative']

        # Collect all valid embeddings and row_ids for this rule
        valid_embeddings = []
        valid_row_ids = []

        for _, row in rule_data.iterrows():
            body = cleaner(str(row['body']))
            row_id = row['row_id']

            if body in text_to_embedding:
                valid_embeddings.append(text_to_embedding[body])
                valid_row_ids.append(row_id)

        if not valid_embeddings:
            continue

        # Convert to numpy array
        query_embeddings = np.array(valid_embeddings)

        # Compute Euclidean distances
        pos_distances = np.linalg.norm(query_embeddings - pos_centroid, axis=1)
        neg_distances = np.linalg.norm(query_embeddings - neg_centroid, axis=1)

        # Score: closer to positive (lower distance) = higher violation score
        rule_predictions = neg_distances - pos_distances

        row_ids.extend(valid_row_ids)
        predictions.extend(rule_predictions)

    print(f"Made predictions for {len(predictions)} test examples")
    return row_ids, np.array(predictions)




def main():
    """Main inference pipeline."""
    print("="*70)
    print("SIMPLE SIMILARITY CLASSIFIER - INFERENCE")
    print("="*70)
    
    # Step 1: Load test data
    test_df = load_test_data()
    
    # Step 2: Load or create fine-tuned model
    print("\n" + "="*50)
    print("MODEL PREPARATION PHASE")
    print("="*50)
    model = load_or_create_finetuned_model(test_df)
    
    # Step 3: Collect all texts
    all_texts = collect_all_texts(test_df)
    
    # Step 4: Generate embeddings with fine-tuned model
    print("\n" + "="*50)
    print("EMBEDDING GENERATION PHASE")
    print("="*50)
    all_embeddings = generate_embeddings(all_texts, model)
    
    # Step 5: Create text to embedding mapping
    text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}
    
    # Step 6: Generate rule embeddings
    rule_embeddings = generate_rule_embeddings(test_df, model)
    
    # Step 7: Create rule centroids from test examples
    rule_centroids = create_rule_centroids(test_df, text_to_embedding, rule_embeddings)
    
    # Step 8: Predict test set
    print("\n" + "="*50)
    print("PREDICTION PHASE")
    print("="*50)
    row_ids, predictions = predict_test_set(test_df, text_to_embedding, rule_centroids)
    
    # Step 9: Create submission with rule-conditioned scores
    submission_df = pd.DataFrame({
        'row_id': row_ids,
        'rule_violation': predictions
    })
    
    submission_df.to_csv('submission_triplet.csv', index=False) #.907 score
    print(f"\nSaved predictions for {len(submission_df)} test examples to submission.csv")
    
    print(f"\n{'='*70}")
    print(f"FINE-TUNED EUCLIDEAN DISTANCE INFERENCE COMPLETED")
    print(f"Model: Fine-tuned BGE on test data triplets")
    print(f"Method: Single centroid with Euclidean distance")
    print(f"Predicted on {len(test_df)} test examples")
    print(f"Prediction stats: min={predictions.min():.4f}, max={predictions.max():.4f}, mean={predictions.mean():.4f}")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()

In [ ]:
#changed aug to 16
!python triplet.py

# Phase 2

In [ ]:
%%writefile constants.py

seed = 0

base_model_path = "/kaggle/input/jigsaw-pretrain-public/pytorch/llama-3.2-3b-instruct/1"
pretrain_lora_path = None
lora_path = "/kaggle/working/pseudo_lora"
use_gptq = "gptq" in base_model_path

positive = "Yes"
negative = "No"
judge_words = "Violation:"
system_prompt = '''You are given a comment from reddit and a rule. 
Your task is to classify whether the comment violates the rule. 
Only respond Yes/No.'''

frac = 0.05# hoping to have 6hrs, .92+ score.
use_train = True

import kagglehub

deterministic = kagglehub.package_import('wasupandceacar/deterministic').deterministic
deterministic.init_all(seed)

In [ ]:
%%writefile utils.py

import numpy as np
import pandas as pd
from datasets import Dataset
from constants import *

def build_prompt(row):
    return f"""{system_prompt}
Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{judge_words} Yes
2) {row["negative_example"]}
{judge_words} No
Comment: {row["body"]}
{judge_words}"""

def get_df():
    merge = list()
    if use_train:
        train_dataset = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/train.csv")
        train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                                "positive_example_1", "positive_example_2", 
                                "negative_example_1", "negative_example_2"]].copy()
        train_df["positive_example"] = np.where(np.random.rand(len(train_df)) < 0.5, train_df["positive_example_1"], train_df["positive_example_2"])
        train_df["negative_example"] = np.where(np.random.rand(len(train_df)) < 0.5, train_df["negative_example_1"], train_df["negative_example_2"])
        train_df.drop(columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"], inplace=True)
        merge.append(train_df)
    test_dataset = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")
    test_dataset = test_dataset.groupby('rule', group_keys=False).apply(lambda x: x.sample(frac=frac, random_state=seed)).reset_index(drop=True)
    print(f"Select {len(test_dataset)} test data")
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[["rule", "subreddit", "positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"]].copy()
            body_col = f"{violation_type}_example_{i}"
            other_positive_col = f"{violation_type}_example_{3-i}"
            sub_dataset["body"] = sub_dataset[body_col]
            sub_dataset[f"{violation_type}_example"] = sub_dataset[other_positive_col]
            anti_violation_type = "negative" if violation_type == "positive" else "positive"
            sub_dataset[f"{anti_violation_type}_example"] = np.where(np.random.rand(len(sub_dataset)) < 0.5, sub_dataset[f"{anti_violation_type}_example_1"], sub_dataset[f"{anti_violation_type}_example_2"])
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            sub_dataset.drop(columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"], inplace=True)
            merge.append(sub_dataset)
    return pd.concat(merge, axis=0).drop_duplicates(ignore_index=True)

def build_dataset(df):
    df["prompt"] = df.apply(build_prompt, axis=1)
    columns = ["prompt"]
    if "rule_violation" in df:
        df["completion"] = df["rule_violation"].map({
            1: positive,
            0: negative,})
        columns.append("completion")
    dataset = Dataset.from_pandas(df[columns])
    return dataset

In [ ]:
%%writefile train.py

import torch
import pandas as pd
from trl import SFTTrainer, SFTConfig
from peft import PeftModel, LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.utils import is_torch_bf16_gpu_available

from utils import *
from constants import *

def main():
    train_dataset = build_dataset(get_df())
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    training_args = SFTConfig(
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        optim="paged_adamw_8bit",
        learning_rate=1e-4,
        weight_decay=0.01,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        fp16=True,
        dataloader_pin_memory=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        save_strategy="no",
        report_to="none",
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )

    if use_gptq:
        model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            device_map="balanced_low_0",
            trust_remote_code=True,
            use_cache=False,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,     
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            ),
            device_map="balanced_low_0",
            trust_remote_code=True,
            use_cache=False,
        )
    tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    tokenizer.pad_token = tokenizer.eos_token
    if pretrain_lora_path:
        model = PeftModel.from_pretrained(model, pretrain_lora_path)
        model = model.merge_and_unload()

    if len(train_dataset) > 0:
        trainer = SFTTrainer(
            model=model,
            processing_class=tokenizer,
            args=training_args,
            train_dataset=train_dataset,
            peft_config=lora_config,
        )
        trainer.train()
        trainer.save_model(lora_path)
    else:
        peft_model = get_peft_model(model, lora_config)
        peft_model.save_pretrained(lora_path)
        tokenizer.save_pretrained(lora_path)

if __name__ == "__main__":
    main()

In [ ]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  bf16:
    enabled: false
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

In [ ]:
%%writefile inference.py

import os
os.environ["VLLM_USE_V1"] = "0"

import random
import vllm
import torch
import numpy as np
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import *
import multiprocessing as mp

def run_inference_on_device(df_slice):
    llm = vllm.LLM(
        base_model_path,
        quantization="gptq" if use_gptq else None,
        tensor_parallel_size=1,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2048,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )
    tokenizer = llm.get_tokenizer()
    outputs = llm.generate(
        build_dataset(df_slice)["prompt"],
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[MultipleChoiceLogitsProcessor(tokenizer, choices=[positive, negative])],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("lora1", 1, lora_path)
    )
    log_probs = [{lp.decoded_token: np.exp(lp.logprob) for lp in out.outputs[0].logprobs[0].values()} for out in outputs]
    predictions = pd.DataFrame(log_probs)[[positive, negative]]
    predictions["row_id"] = df_slice["row_id"].values
    return predictions

def worker(device_id, df_slice, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")
    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds

def main():
    test_df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")
    test_df["positive_example"] = test_df.apply(lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]), axis=1)
    test_df["negative_example"] = test_df.apply(lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]), axis=1)
    test_df = test_df.drop(columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"], errors="ignore")

    mid = len(test_df) // 2
    df0 = test_df.iloc[:mid].reset_index(drop=True)
    df1 = test_df.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()
    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)
    submission = predictions[["row_id", positive]].rename(columns={positive: "rule_violation"})
    submission.to_csv("/kaggle/working/submission_llama.csv", index=False) #.916  score

if __name__ == "__main__":
    main()

In [ ]:
# changed frac back to .05
!accelerate launch --config_file accelerate_config.yaml train.py
!python inference.py

# Phase 3

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup

# %env KAGGLE_IS_COMPETITION_RERUN ='true'
EPOCHS = 8

env: KAGGLE_IS_COMPETITION_RERUN='true'


In [2]:
MODELS = [
    # (256, "/kaggle/input/deberta-base-923-pseudo/model_pseudo_deberta_base_seed33.bin"),
    # (256, "/kaggle/input/pretrained-model/pretrained_model.bin"),
    # (256, "/kaggle/input/3-256l-1200k-920-wbce/pretrained_model.bin"),
    # (128, "/kaggle/input/42-123-700k-128l-923-jigsaw/model_seed_123_pseudo.bin"),
    # (128, "/kaggle/input/42-123-700k-128l-923-jigsaw/model_seed_42_pseudo.bin"),
    (256,'/kaggle/input/deberta_large_seed333_jigsaw/pytorch/default/2/model_pseudo_deberta_large_seed333.bin'),
    (256,'/kaggle/input/deberta_large_seed333_jigsaw/pytorch/default/2/model_pseudo_deberta_large_seed333.bin'),

    # (256,'/kaggle/input/deberta_large_seed333_jigsaw/pytorch/default/1/model_pseudo_deberta_large_seed333.bin')

]

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2

MODEL_PATH= '/kaggle/input/deberta-v3-large/transformers/default/1/deberta-v3-large'
# MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"

# Generate seeds for all models
random.seed(300)
ALL_SEEDS = [random.randint(1, 10000) for _ in range(len(MODELS) * 2)]
print(f"Generated seeds: {ALL_SEEDS}")

Generated seeds: [9793, 5840]


In [3]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [4]:
df = pd.read_csv(train_path)
df['rule']= df['rule'].str.lower().str.strip()
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [5]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [6]:
test_df = pd.read_csv(test_path)
test_df['rule'] = test_df.rule.str.lower().str.strip()

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts = df.text.tolist() + augmented_train[0] + augmented_test[0]
augmented_labels = df.label.tolist() + augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before: {augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:', augmented_df.shape)
augmented_df['rule'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

augmented_df.head()

Before: (10185, 2)
After: (1875, 2)


,text,label,rule,body
0,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ..."
1,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...
2,"no advertising: spam, referral links, unsolici...",0.0,"no advertising: spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...
3,"no advertising: spam, referral links, unsolici...",0.0,"no advertising: spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...
4,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [8]:
# Create rule_id mapping
rule_map = {rule: idx for idx, rule in enumerate(augmented_df.rule.unique())}
augmented_df['rule_id'] = augmented_df.rule.map(rule_map)

print(f"Total samples: {len(augmented_df)}")
print(f"Unique rules: {len(rule_map)}")

Total samples: 1875
Unique rules: 2


In [9]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len,weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids
        self.weights = weights if weights is not None else [1.0]*len(texts)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        item['weights'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

In [10]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def get_optimizer_grouped_parameters(model, lr=1e-5, weight_decay=0.01, llrd_factor=0.95):
    """
    Layer-wise learning rate decay.
    Top layers get higher learning rates, lower layers get progressively smaller rates.
    
    Args:
        model: JigsawModel instance
        lr: Learning rate for the top layer (classifier head)
        weight_decay: Weight decay for regularization
        llrd_factor: Decay factor (0.95 means each lower layer gets 95% of the layer above)
    
    Returns:
        List of parameter groups for optimizer
    """
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    
    # Get number of layers
    num_layers = model.base.config.num_hidden_layers  # 24 for DeBERTa-large
    
    optimizer_grouped_parameters = []
    
    # Classifier head (highest learning rate = lr)
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.named_parameters() 
                  if "base" not in n and not any(nd in n for nd in no_decay) and p.requires_grad],
        "weight_decay": weight_decay,
        "lr": lr,
    })
    
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.named_parameters() 
                  if "base" not in n and any(nd in n for nd in no_decay) and p.requires_grad],
        "weight_decay": 0.0,
        "lr": lr,
    })
    
    # Encoder layers (from top to bottom, decaying learning rate)
    for layer_num in range(num_layers - 1, -1, -1):  # 23, 22, ..., 1, 0
        # Calculate LR: top layer (23) gets lr*llrd_factor^1, layer 0 gets lr*llrd_factor^24
        layer_lr = lr * (llrd_factor ** (num_layers - layer_num))
        
        # Parameters with weight decay
        optimizer_grouped_parameters.append({
            "params": [p for n, p in model.named_parameters() 
                      if f'encoder.layer.{layer_num}.' in n 
                      and not any(nd in n for nd in no_decay) 
                      and p.requires_grad],
            "weight_decay": weight_decay,
            "lr": layer_lr,
        })
        
        # Parameters without weight decay (bias, LayerNorm)
        optimizer_grouped_parameters.append({
            "params": [p for n, p in model.named_parameters() 
                      if f'encoder.layer.{layer_num}.' in n 
                      and any(nd in n for nd in no_decay) 
                      and p.requires_grad],
            "weight_decay": 0.0,
            "lr": layer_lr,
        })
    
    # Encoder-level parameters (rel_embeddings, LayerNorm, etc.) - use middle-range LR
    encoder_level_lr = lr * (llrd_factor ** (num_layers // 2))  # Middle layer LR
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.named_parameters() 
                  if 'encoder' in n and 'encoder.layer' not in n 
                  and not any(nd in n for nd in no_decay)
                  and p.requires_grad],
        "weight_decay": weight_decay,
        "lr": encoder_level_lr,
    })
    
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.named_parameters() 
                  if 'encoder' in n and 'encoder.layer' not in n 
                  and any(nd in n for nd in no_decay)
                  and p.requires_grad],
        "weight_decay": 0.0,
        "lr": encoder_level_lr,
    })
    
    # Embeddings are frozen, so we don't add them
    # (they have requires_grad=False set in train_model_seed)
    
    return optimizer_grouped_parameters

In [11]:
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler):
  model.train()
  total_loss = 0
  optimizer.zero_grad()  # Zero gradients at start

  for batch_idx, batch in enumerate(tqdm(loader, desc='Training')):
      input_ids = batch["input_ids"].to(device)
      mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      weights = batch["weights"].to(device)

      with torch.cuda.amp.autocast():
          logits = model(input_ids, mask)
          loss = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
          loss = (loss * weights).mean()
          loss = loss / GRADIENT_ACCUMULATION_STEPS  # Scale loss

      scaler.scale(loss).backward()

      # Only step every GRADIENT_ACCUMULATION_STEPS or at the end
      if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(loader):
          scaler.unscale_(optimizer)
          torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
          scaler.step(optimizer)
          scaler.update()
          optimizer.zero_grad()

          if scheduler:
              scheduler.step()

      total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS  # Unscale for logging

  return total_loss / len(loader)

In [12]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [ ]:
def create_train_val_splits(seed, augmented_df):
    train_data, val_data = train_test_split(
        augmented_df, 
        test_size=0.2, 
        stratify=augmented_df["rule"], 
        random_state=seed
    )
    train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
    val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
    print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}')

def train_model_seed(seed, gpu_id, max_length, pretrained_path, model_idx):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Model {model_idx}, Seed {seed}] Training on {device} with max_length={max_length}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist(), 
        train_data['label'].tolist(), 
        train_data['rule_id'].tolist(), 
        tokenizer, max_length,
        [1.0]*len(train_data)
    )

    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, max_length
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    scaler = torch.cuda.amp.GradScaler()
    model = JigsawModel(MODEL_PATH).to(device)
    
    # Load pretrained weights with auto-detection of DataParallel
    state_dict = torch.load(pretrained_path, map_location=device)
    
    # Auto-detect and strip 'module.' prefix if present
    if any(key.startswith('module.') for key in state_dict.keys()):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        print(f"[Model {model_idx}, Seed {seed}] Stripped 'module.' prefix from state dict")
    
    model.load_state_dict(state_dict)
    
    # Freeze embeddings
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    # Use LLRD with decay factor 0.95
    grouped_params = get_optimizer_grouped_parameters(
        model, 
        lr=1e-5,  # Top layer learning rate
        weight_decay=0.01,
        llrd_factor=0.95  # Each lower layer gets 95% of the layer above
    )
    optimizer = torch.optim.AdamW(grouped_params, eps=1e-6)
    # total_steps = EPOCHS * len(train_loader)
    total_steps = EPOCHS * (len(train_loader) // GRADIENT_ACCUMULATION_STEPS)

    warmup_steps = int(0.05 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss = None
    for epoch in range(EPOCHS):
        print(f"[Model {model_idx}, Seed {seed}] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, scaler)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Model {model_idx}, Seed {seed}] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss = val_loss
            torch.save(model.state_dict(), f"model_{model_idx}_seed_{seed}.bin")
    
    print(f"[Model {model_idx}, Seed {seed}] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_model_{model_idx}_seed_{seed}.json', 'w') as f:
        json.dump({'model_idx': model_idx, 'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return model_idx, seed, best_auc

In [14]:
def inference_model_seed(seed, gpu_id, max_length, model_idx, df_test):
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Model {model_idx}, Seed {seed}] Inference on {device} with max_length={max_length}")
    
    # Create test dataset with appropriate max_length
    test_ds = JigsawDataset(
        df_test['text'].tolist(), 
        [0]*len(df_test), 
        [0]*len(df_test), 
        tokenizer, 
        max_length
    )
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
    # Load trained model
    model = JigsawModel(MODEL_PATH).to(device)
    model.load_state_dict(torch.load(f"model_{model_idx}_seed_{seed}.bin", map_location=device))
    model.eval()
    
    # Generate predictions
    test_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Model {model_idx}, Seed {seed}"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits = model(ids, mask)
            test_preds.extend(torch.sigmoid(logits).cpu().numpy())
    
    # Save predictions
    test_preds = np.array(test_preds)
    np.save(f'predictions_model_{model_idx}_seed_{seed}.npy', test_preds)
    print(f"[Model {model_idx}, Seed {seed}] Saved predictions_model_{model_idx}_seed_{seed}.npy")
    
    return model_idx, seed

In [15]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    mp.set_start_method('fork', force=True)
    
    all_results = []
    
    # Iterate through models sequentially
    for model_idx, (max_length, pretrained_path) in enumerate(MODELS):
        print(f"\n{'='*60}")
        print(f"Processing Model {model_idx}: max_length={max_length}")
        print(f"Pretrained path: {pretrained_path}")
        print(f"{'='*60}\n")
        
        # Get the 2 seeds for this model
        seed1 = ALL_SEEDS[model_idx * 2]
        seed2 = ALL_SEEDS[model_idx * 2 + 1]
        
        print(f"Using seeds: {seed1}, {seed2}")
        
        # Create train/val splits for both seeds
        create_train_val_splits(seed1, augmented_df)
        create_train_val_splits(seed2, augmented_df)
        
        # Train both seeds in parallel (GPU 0 and GPU 1)
        processes = []
        
        p1 = mp.Process(target=train_model_seed, args=(seed1, 0, max_length, pretrained_path, model_idx))
        p1.start()
        processes.append(p1)
        
        p2 = mp.Process(target=train_model_seed, args=(seed2, 1, max_length, pretrained_path, model_idx))
        p2.start()
        processes.append(p2)
        
        # Wait for both to complete
        for p in processes:
            p.join()
        
        # Collect results
        import json
        for seed in [seed1, seed2]:
            with open(f'results_model_{model_idx}_seed_{seed}.json', 'r') as f:
                all_results.append(json.load(f))
        
        print(f"\nModel {model_idx} training completed!")
    
    # Print summary
    print(f"\n{'='*60}")
    print("Training Summary")
    print(f"{'='*60}")
    for result in all_results:
        print(f"Model {result['model_idx']}, Seed {result['seed']}: "
              f"AUC={result['best_auc']:.4f}, Loss={result['best_loss']:.4f}")
    
    print("\nAll models trained!")


Processing Model 0: max_length=256
Pretrained path: /kaggle/input/deberta_large_seed333_jigsaw/pytorch/default/2/model_pseudo_deberta_large_seed333.bin

Using seeds: 9793, 5840
Seed 9793 splits saved: train=1500, val=375
Seed 5840 splits saved: train=1500, val=375
[Model 0, Seed 5840] Training on cuda:1 with max_length=256
[Model 0, Seed 9793] Training on cuda:0 with max_length=256


/tmp/ipykernel_36/3282834077.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_36/3282834077.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
2025-10-23 16:04:35.128953: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-23 16:04:35.128953: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761235475.356228      86 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E

[Model 0, Seed 5840] Epoch 1/10[Model 0, Seed 9793] Epoch 1/10



Training:   0%|          | 0/188 [00:00<?, ?it/s]/tmp/ipykernel_36/2180942622.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_36/2180942622.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training: 100%|██████████| 188/188 [01:41<00:00,  1.85it/s]


[Model 0, Seed 9793] Loss: 0.4260, Val Loss: 0.4107, Val AUC: 0.8941
[Model 0, Seed 9793] Epoch 2/10


...

[Model 0, Seed 9793] Loss: 0.0282, Val Loss: 1.3459, Val AUC: 0.8556
[Model 0, Seed 9793] Epoch 10/10


Training:   0%|          | 0/188 [00:00<?, ?it/s]/tmp/ipykernel_36/2180942622.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training:  14%|█▍        | 27/188 [00:14<01:27,  1.85it/s]

[Model 0, Seed 5840] Loss: 0.0329, Val Loss: 1.2248, Val AUC: 0.8593
[Model 0, Seed 5840] Epoch 10/10


Training:   0%|          | 0/188 [00:00<?, ?it/s]/tmp/ipykernel_36/2180942622.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training: 100%|██████████| 188/188 [01:45<00:00,  1.79it/s]


[Model 0, Seed 9793] Loss: 0.0173, Val Loss: 1.3956, Val AUC: 0.8569
[Model 0, Seed 9793] Best validation AUC: 0.8941
[Model 0, Seed 5840] Loss: 0.0285, Val Loss: 1.2618, Val AUC: 0.8590
[Model 0, Seed 5840] Best validation AUC: 0.8932

Model 0 training completed!

Training Summary
Model 0, Seed 9793: AUC=0.8941, Loss=0.4107
Model 0, Seed 5840: AUC=0.8932, Loss=0.4372

All models trained!


In [16]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    
    # Load test data
    df_test = pd.read_csv(test_path)
    df_test['rule'] = df_test['rule'].str.lower().str.strip()
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    # Iterate through models sequentially for inference
    for model_idx, (max_length, pretrained_path) in enumerate(MODELS):
        print(f"\n{'='*60}")
        print(f"Inference Model {model_idx}: max_length={max_length}")
        print(f"{'='*60}\n")
        
        # Get the 2 seeds for this model
        seed1 = ALL_SEEDS[model_idx * 2]
        seed2 = ALL_SEEDS[model_idx * 2 + 1]
        
        print(f"Using seeds: {seed1}, {seed2}")
        
        # Run inference for both seeds in parallel (GPU 0 and GPU 1)
        processes = []
        
        p1 = mp.Process(target=inference_model_seed, args=(seed1, 0, max_length, model_idx, df_test))
        p1.start()
        processes.append(p1)
        
        p2 = mp.Process(target=inference_model_seed, args=(seed2, 1, max_length, model_idx, df_test))
        p2.start()
        processes.append(p2)
        
        # Wait for both to complete
        for p in processes:
            p.join()
        
        print(f"\nModel {model_idx} inference completed!")
    
    # Load all predictions and ensemble
    print(f"\n{'='*60}")
    print("Creating Ensemble")
    print(f"{'='*60}\n")
    
    all_predictions = []
    for model_idx, (max_length, pretrained_path) in enumerate(MODELS):
        seed1 = ALL_SEEDS[model_idx * 2]
        seed2 = ALL_SEEDS[model_idx * 2 + 1]
        
        for seed in [seed1, seed2]:
            pred_path = f'predictions_model_{model_idx}_seed_{seed}.npy'
            preds = np.load(pred_path)
            all_predictions.append(preds)
            print(f"Loaded {pred_path}")
    
    # Ensemble: average all predictions
    ensemble_preds = np.mean(all_predictions, axis=0)
    
    # Create submission
    sample = pd.read_csv(sample_sub_path)
    sample["rule_violation"] = ensemble_preds
    sample.to_csv("submission_deberta.csv", index=False) #.925 score. 
    
    print(f"\nEnsemble complete! Averaged {len(all_predictions)} models")
    print(f"Submission saved to submission.csv")
    
else:
    !touch submission.csv


Inference Model 0: max_length=256

Using seeds: 9793, 5840
[Model 0, Seed 9793] Inference on cuda:0 with max_length=256
[Model 0, Seed 5840] Inference on cuda:1 with max_length=256


2025-10-23 16:26:58.621440: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-23 16:26:58.621440: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761236818.644189    7974 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761236818.644189    7971 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761236818.651473    7974 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1761236818.651473    7971 cuda_blas.cc:1

[Model 0, Seed 9793] Saved predictions_model_0_seed_9793.npy


Model 0, Seed 5840: 100%|██████████| 2/2 [00:01<00:00,  1.67it/s]


[Model 0, Seed 5840] Saved predictions_model_0_seed_5840.npy

Model 0 inference completed!

Creating Ensemble

Loaded predictions_model_0_seed_9793.npy
Loaded predictions_model_0_seed_5840.npy

Ensemble complete! Averaged 2 models
Submission saved to submission.csv


In [17]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sub3= pd.read_csv('submission_triplet.csv') #.907
    sub2= pd.read_csv('submission_deberta.csv') #.924
    sub1= pd.read_csv('submission_llama.csv')   #.916
    
    sub1['llm']= sub1['rule_violation'].rank(method='average')/(len(sub1)+1)
    sub2['bert']= sub2['rule_violation'].rank(method='average')/(len(sub2)+1)
    sub3['triplet']= sub3['rule_violation'].rank(method='average')/(len(sub3)+1)
    
    sub= pd.merge(sub2[['row_id','bert']],sub1[['row_id','llm']],on='row_id',how='left')
    sub= sub.merge(sub3[['row_id','triplet']],on='row_id',how='left')
    
    sub['rule_violation']= .5*sub['bert']+ .3*sub['llm']+ .2*sub['triplet']
    
    sub[['row_id','rule_violation']].to_csv('submission.csv',index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'submission_triplet.csv'

In [ ]:
!head submission.csv

In [ ]:
#important model, what lr to use, what additional models can be added, what techniques can be done